# Chunking Implementation

## Objective
To divide document text into smaller chunks and compare different
chunking techniques used in Generative AI and Retrieval-Augmented
Generation (RAG).

## Techniques
1. Fixed-Size Chunking
2. Overlapping Chunking
3. Sentence-Based Chunking
4. Paragraph-Based Chunking
5. Semantic Chunking

## Evaluation
Compare chunk count, chunk length, context preservation, and
the advantages and limitations of each technique.

### Importing Required Libraries and Configuring Logging

This cell imports the Python libraries required for the chunking implementation and configures logging to display useful information during execution.

- **`logging`** – Displays informational messages and helps track the execution of the program.
- **`re`** – Provides regular expression operations, which are used to split text into sentences and paragraphs.
- **`List`** – Provides a type hint for lists. It can be removed if it is not used in the code.
- **`logging.basicConfig()`** – Configures the logging level and output format.
- **`level=logging.INFO`** – Displays informational messages, warnings, and errors.
- **`format`** – Specifies the timestamp, log level, logger name, and message format.
- **`datefmt`** – Defines the format of the timestamp.
- **`force=True`** – Replaces existing logging configuration.
- **`logging.getLogger("chunking")`** – Creates or retrieves a logger named `chunking`.
- **`logger.info()`** – Prints an informational log message confirming that the chunking implementation has started.

This cell prepares the utilities required by the subsequent chunking functions.

In [1]:
import logging
import re
from typing import List

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    force=True
)

logger = logging.getLogger("chunking")

logger.info("Chunking implementation initialized.")

2026-10-21 17:57:48 | INFO | chunking | Chunking implementation initialized.


### Preparing the Input Document

This cell creates a sample policy document that will be used to test the different chunking techniques.

- **`document`** – Stores the Annual Leave Policy, Sick Leave Policy, and Work From Home Policy as a multi-line string.
- **`.strip()`** – Removes whitespace from the beginning and end of the document.
- **`len(document)`** – Calculates the total number of characters in the document.
- **`document.split()`** – Separates the text using whitespace so that the number of words can be counted.
- **`document.splitlines()`** – Separates the document into lines.
- **`type(document).__name__`** – Displays the data type of the document.
- **`print(document)`** – Displays the complete document before chunking.
- **`if document.strip()`** – Checks whether the document contains meaningful text.

The sample document contains 619 characters, 93 whitespace-separated words, and 17 lines.

Using the same document for every technique makes it easier to compare the results fairly.

In [2]:
document = """
Annual Leave Policy

Employees receive 20 days of annual leave each year.
Leave requests must be approved by their managers.
Employees should submit leave requests five working days in advance.

Sick Leave Policy

Employees can request sick leave when they are unwell.
A medical certificate may be required for extended absence.
Employees must inform their managers when taking sick leave.

Work From Home Policy

Eligible employees may work from home with manager approval.
Employees must attend scheduled meetings and remain available during working hours.
Company data must be handled securely when working remotely.
""".strip()

print("=" * 70)
print("STEP 1: INPUT DOCUMENT PREPARATION")
print("=" * 70)

print("\n[1] Document loaded successfully.")
print("[2] Document type:", type(document).__name__)
print("[3] Total characters:", len(document))
print("[4] Total words:", len(document.split()))
print("[5] Total lines:", len(document.splitlines()))

print("\n[6] Document content:")
print("-" * 70)
print(document)
print("-" * 70)

print("\n[7] Input validation:")
if document.strip():
    print("PASS: Document contains text and is ready for chunking.")
else:
    print("FAIL: Document is empty.")

print("\nInput preparation completed.")

STEP 1: INPUT DOCUMENT PREPARATION

[1] Document loaded successfully.
[2] Document type: str
[3] Total characters: 619
[4] Total words: 93
[5] Total lines: 17

[6] Document content:
----------------------------------------------------------------------
Annual Leave Policy

Employees receive 20 days of annual leave each year.
Leave requests must be approved by their managers.
Employees should submit leave requests five working days in advance.

Sick Leave Policy

Employees can request sick leave when they are unwell.
A medical certificate may be required for extended absence.
Employees must inform their managers when taking sick leave.

Work From Home Policy

Eligible employees may work from home with manager approval.
Employees must attend scheduled meetings and remain available during working hours.
Company data must be handled securely when working remotely.
----------------------------------------------------------------------

[7] Input validation:
PASS: Document contains text and 

### Implementing Fixed-Size Chunking

This cell defines a function that divides a document into chunks containing a specified maximum number of characters.

- **`fixed_size_chunking(text, chunk_size=100)`** – Defines the function with a default chunk size of 100 characters.
- **`isinstance(text, str)`** – Checks whether the input is a string.
- **`raise TypeError(...)`** – Stops execution if the input has an invalid data type.
- **`chunk_size <= 0`** – Checks that the chunk size is positive.
- **`text.strip()`** – Removes leading and trailing whitespace.
- **`if not text`** – Handles an empty document by returning an empty list.
- **`total_characters = len(text)`** – Calculates the document length.
- **`(total_characters + chunk_size - 1) // chunk_size`** – Calculates the number of chunks, including a possible final partial chunk.
- **`chunks = []`** – Creates a list to store the generated chunks.
- **`range(0, total_characters, chunk_size)`** – Generates the starting positions for the chunks.
- **`min(start + chunk_size, total_characters)`** – Prevents the ending position from exceeding the document length.
- **`text[start:end]`** – Extracts a section of text using string slicing.
- **`chunks.append(chunk)`** – Adds each chunk to the list.
- **`"".join(chunks)`** – Combines the chunks to check whether the original text is preserved.
- **`all(...)`** – Checks that each chunk is non-empty and within the size limit.
- **`return chunks`** – Returns the completed list of chunks.

Fixed-size chunking is simple and predictable, but it can split words and sentences across chunk boundaries.

In [3]:
def fixed_size_chunking(text, chunk_size=100):
    print("=" * 70)
    print("STEP 2: FIXED-SIZE CHUNKING")
    print("=" * 70)

    # 1. Validate the input
    print("\n[1] VALIDATING INPUT")

    if not isinstance(text, str):
        raise TypeError("Input must be a string.")

    if chunk_size <= 0:
        raise ValueError("Chunk size must be greater than zero.")

    text = text.strip()

    if not text:
        print("Input is empty. No chunks will be created.")
        return []

    print("Input validation: PASSED")
    print(f"Total input characters: {len(text)}")
    print(f"Selected chunk size: {chunk_size} characters")
    print("Overlap: 0 characters")

    # 2. Calculate the expected number of chunks
    total_characters = len(text)
    total_chunks = (total_characters + chunk_size - 1) // chunk_size

    print("\n[2] CALCULATING CHUNKS")
    print(f"Calculation: {total_characters} / {chunk_size}")
    print(f"Expected number of chunks: {total_chunks}")
    print("Each chunk contains at most", chunk_size, "characters.")

    # 3. Split the text
    chunks = []

    print("\n[3] SPLITTING THE DOCUMENT")
    print("-" * 70)

    for start in range(0, total_characters, chunk_size):
        end = min(start + chunk_size, total_characters)
        chunk = text[start:end]
        chunks.append(chunk)

        print(f"\nChunk {len(chunks)}")
        print(f"Character positions: {start} to {end - 1}")
        print(f"Chunk length: {len(chunk)} characters")
        print(f"Chunk content: {chunk!r}")

    # 4. Validate the results
    print("\n[4] VALIDATING RESULTS")
    print("-" * 70)

    combined_text = "".join(chunks)
    all_text_preserved = combined_text == text
    all_chunks_valid = all(
        0 < len(chunk) <= chunk_size
        for chunk in chunks
    )

    print(f"Total chunks created: {len(chunks)}")
    print(f"All chunks within size limit: {all_chunks_valid}")
    print(f"All original text preserved: {all_text_preserved}")
    print(f"Overlap between chunks: None")

    if all_text_preserved and all_chunks_valid:
        print("\nRESULT: Fixed-size chunking completed successfully.")
    else:
        print("\nRESULT: Validation failed. Check the implementation.")

    return chunks

### Executing Fixed-Size Chunking

This cell executes the fixed-size chunking function using the sample document.

- **`document`** – Provides the text to be split.
- **`chunk_size=100`** – Sets the maximum chunk length to 100 characters.
- **`fixed_size_chunking(...)`** – Runs the function defined in the previous cell.
- **`fixed_chunks`** – Stores the returned list of chunks for later experiments.

The document should produce seven chunks: six chunks containing 100 characters and a final chunk containing 19 characters.

The function also displays the chunk lengths and checks whether the text has been preserved.

This cell executes the function; the previous cell only defined it.

In [4]:
fixed_chunks = fixed_size_chunking(
    document,
    chunk_size=100
)

STEP 2: FIXED-SIZE CHUNKING

[1] VALIDATING INPUT
Input validation: PASSED
Total input characters: 619
Selected chunk size: 100 characters
Overlap: 0 characters

[2] CALCULATING CHUNKS
Calculation: 619 / 100
Expected number of chunks: 7
Each chunk contains at most 100 characters.

[3] SPLITTING THE DOCUMENT
----------------------------------------------------------------------

Chunk 1
Character positions: 0 to 99
Chunk length: 100 characters
Chunk content: 'Annual Leave Policy\n\nEmployees receive 20 days of annual leave each year.\nLeave requests must be app'

Chunk 2
Character positions: 100 to 199
Chunk length: 100 characters
Chunk content: 'roved by their managers.\nEmployees should submit leave requests five working days in advance.\n\nSick '

Chunk 3
Character positions: 200 to 299
Chunk length: 100 characters
Chunk content: 'Leave Policy\n\nEmployees can request sick leave when they are unwell.\nA medical certificate may be re'

Chunk 4
Character positions: 300 to 399
Chunk len

### Implementing Overlapping Chunking

This cell defines a function that creates chunks with shared text between consecutive chunks.

- **`chunk_size=100`** – Sets the maximum length of each chunk.
- **`overlap=40`** – Specifies the number of characters shared between consecutive full-size chunks.
- **Input validation** – Checks the input type, positive chunk size, and valid overlap.
- **`text.strip()`** – Removes leading and trailing whitespace.
- **`step_size = chunk_size - overlap`** – Calculates how far the next chunk starts after the previous chunk's starting position.
- **`chunks = []`** – Stores the generated chunks.
- **`start = 0`** – Sets the initial starting position.
- **`while start < len(text)`** – Continues creating chunks until the document's end is reached.
- **`text[start:end]`** – Extracts the current chunk.
- **`start += step_size`** – Advances the starting position for the next chunk.
- **`if end == len(text): break`** – Stops when the final chunk reaches the end of the document.
- **Reconstruction logic** – Removes the repeated overlap from subsequent chunks before checking whether the original text can be recovered.

For a chunk size of 100 and an overlap of 40, the step size is 60 characters.

Overlap can preserve context across chunk boundaries, but it also increases redundancy and the total number of characters stored.

In [5]:
def overlapping_chunking(text, chunk_size=100, overlap=40):
    print("=" * 70)
    print("STEP 3: OVERLAPPING CHUNKING")
    print("=" * 70)

    print("\n[1] VALIDATING INPUT")

    if not isinstance(text, str):
        raise TypeError("Input must be a string.")

    if chunk_size <= 0:
        raise ValueError("Chunk size must be greater than zero.")

    if overlap < 0 or overlap >= chunk_size:
        raise ValueError("Overlap must be >= 0 and < chunk size.")

    text = text.strip()

    if not text:
        print("Input is empty. No chunks will be created.")
        return []

    print("Input validation: PASSED")
    print(f"Total characters: {len(text)}")
    print(f"Chunk size: {chunk_size}")
    print(f"Overlap: {overlap}")

    step_size = chunk_size - overlap

    print("\n[2] CALCULATING STEP SIZE")
    print(f"Step size = {chunk_size} - {overlap} = {step_size}")

    chunks = []
    start = 0

    print("\n[3] CREATING OVERLAPPING CHUNKS")
    print("-" * 70)

    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunk = text[start:end]
        chunks.append(chunk)

        print(f"\nChunk {len(chunks)}")
        print(f"Character positions: {start} to {end - 1}")
        print(f"Chunk length: {len(chunk)} characters")
        print(f"Chunk content: {chunk!r}")

        # Stop if the end of the document has been reached.
        if end == len(text):
            break

        start += step_size

    print("\n[4] VALIDATING RESULTS")
    print("-" * 70)

    # Reconstruct the original text by removing repeated overlap.
    reconstructed = chunks[0]

    for i in range(1, len(chunks)):
        reconstructed += chunks[i][overlap:]

    text_preserved = reconstructed == text
    sizes_valid = all(
        0 < len(chunk) <= chunk_size for chunk in chunks
    )

    print(f"Total chunks created: {len(chunks)}")
    print(f"All chunks within size limit: {sizes_valid}")
    print(f"Original text reconstructed correctly: {text_preserved}")

    print("\nRESULT: Overlapping chunking completed.")

    return chunks

### Executing Overlapping Chunking

This cell applies overlapping chunking to the sample document.

- **`document`** – Supplies the input text.
- **`chunk_size=100`** – Sets the maximum chunk length.
- **`overlap=40`** – Configures the shared text between consecutive chunks.
- **`overlapping_chunks`** – Stores the generated chunks for later comparisons.

The function displays the chunk contents, character positions, lengths, step size, and validation results.

The document should produce ten chunks with these settings. Consecutive full-size chunks should share 40 characters.

This cell demonstrates how overlapping chunks retain some context from the previous chunk.

In [6]:
overlapping_chunks = overlapping_chunking(
    document,
    chunk_size=100,
    overlap=40
)

STEP 3: OVERLAPPING CHUNKING

[1] VALIDATING INPUT
Input validation: PASSED
Total characters: 619
Chunk size: 100
Overlap: 40

[2] CALCULATING STEP SIZE
Step size = 100 - 40 = 60

[3] CREATING OVERLAPPING CHUNKS
----------------------------------------------------------------------

Chunk 1
Character positions: 0 to 99
Chunk length: 100 characters
Chunk content: 'Annual Leave Policy\n\nEmployees receive 20 days of annual leave each year.\nLeave requests must be app'

Chunk 2
Character positions: 60 to 159
Chunk length: 100 characters
Chunk content: 've each year.\nLeave requests must be approved by their managers.\nEmployees should submit leave reque'

Chunk 3
Character positions: 120 to 219
Chunk length: 100 characters
Chunk content: 'ers.\nEmployees should submit leave requests five working days in advance.\n\nSick Leave Policy\n\nEmploy'

Chunk 4
Character positions: 180 to 279
Chunk length: 100 characters
Chunk content: 's in advance.\n\nSick Leave Policy\n\nEmployees can request s

### Implementing Sentence-Based Chunking

This cell defines a function that splits a document into sentence-like units.

- **`text`** – Represents the input document.
- **`isinstance(text, str)`** – Validates the input data type.
- **`text.strip()`** – Removes leading and trailing whitespace.
- **`re.split(r'(?<=[.!?])\s+', text)`** – Splits the text after sentence-ending punctuation followed by whitespace.
- **`(?<=[.!?])`** – Uses a positive lookbehind to check for a period, exclamation mark, or question mark without consuming that punctuation.
- **`\s+`** – Matches one or more whitespace characters.
- **The resulting list** – Contains the sentence-like units used as chunks.
- **Validation and print statements** – Display the number of chunks, their lengths, their content, and whether they are non-empty.

Sentence-based chunking helps keep individual statements together instead of splitting text at arbitrary character positions.

The regular expression is a basic sentence splitter. Policy headings without sentence-ending punctuation may remain attached to the following sentence, and complex punctuation may not be handled correctly.

In [7]:
def sentence_based_chunking(text):
    print("=" * 70)
    print("STEP 4: SENTENCE-BASED CHUNKING")
    print("=" * 70)

    print("\n[1] VALIDATING INPUT")

    if not isinstance(text, str):
        raise TypeError("Input must be a string.")

    text = text.strip()

    if not text:
        print("Input is empty. No chunks will be created.")
        return []

    print("Input validation: PASSED")
    print(f"Total characters: {len(text)}")

    print("\n[2] IDENTIFYING SENTENCES")

    # Split after sentence-ending punctuation followed by whitespace.
    sentences = re.split(r'(?<=[.!?])\s+', text)

    sentences = [
        sentence.strip()
        for sentence in sentences
        if sentence.strip()
    ]

    print(f"Total sentences identified: {len(sentences)}")

    for i, sentence in enumerate(sentences, start=1):
        print(f"\nSentence {i}:")
        print(f"Length: {len(sentence)} characters")
        print(f"Text: {sentence}")

    print("\n[3] CREATING SENTENCE CHUNKS")

    # Initially, each sentence becomes one chunk.
    chunks = sentences.copy()

    for i, chunk in enumerate(chunks, start=1):
        print(f"\nChunk {i}:")
        print(f"Length: {len(chunk)} characters")
        print(f"Content: {chunk}")

    print("\n[4] VALIDATING RESULTS")

    print(f"Total chunks created: {len(chunks)}")
    print(f"All chunks are non-empty: {all(chunks)}")
    print("Note: This method preserves sentence boundaries for this sample.")

    return chunks

### Executing Sentence-Based Chunking

This cell runs the sentence-based chunking function on the sample document.

- **`sentence_based_chunking(document)`** – Splits the document into sentence-like units.
- **`sentence_chunks`** – Stores the resulting list of chunks for use in later experiments.

The sample document should produce nine chunks. Some chunks contain a policy heading followed by its first sentence because the heading does not end with sentence punctuation.

This cell creates the sentence-based chunks used in the comparison table and retrieval simulation.

In [8]:
sentence_chunks = sentence_based_chunking(document)

STEP 4: SENTENCE-BASED CHUNKING

[1] VALIDATING INPUT
Input validation: PASSED
Total characters: 619

[2] IDENTIFYING SENTENCES
Total sentences identified: 9

Sentence 1:
Length: 73 characters
Text: Annual Leave Policy

Employees receive 20 days of annual leave each year.

Sentence 2:
Length: 50 characters
Text: Leave requests must be approved by their managers.

Sentence 3:
Length: 68 characters
Text: Employees should submit leave requests five working days in advance.

Sentence 4:
Length: 73 characters
Text: Sick Leave Policy

Employees can request sick leave when they are unwell.

Sentence 5:
Length: 59 characters
Text: A medical certificate may be required for extended absence.

Sentence 6:
Length: 60 characters
Text: Employees must inform their managers when taking sick leave.

Sentence 7:
Length: 83 characters
Text: Work From Home Policy

Eligible employees may work from home with manager approval.

Sentence 8:
Length: 83 characters
Text: Employees must attend scheduled meetings 

### Implementing Paragraph-Based Chunking

This cell defines a function that splits a document at blank-line boundaries.

- **`text`** – Represents the input document.
- **Input validation** – Checks the input type and handles empty text.
- **`re.split(r"\n\s*\n", text)`** – Splits the document wherever blank lines separate sections.
- **The list comprehension** – Removes empty sections and strips whitespace from each paragraph.
- **`chunks = []`** – Creates a list to store the paragraphs.
- **`enumerate(paragraphs, start=1)`** – Iterates through the paragraphs and assigns each one a readable number.
- **`chunks.append(paragraph)`** – Stores each paragraph as a chunk.
- **`all(chunks)`** – Checks whether all chunks are non-empty.
- **`"\n\n".join(chunks)`** – Reconstructs the document using blank lines between chunks.
- **`reconstructed == text`** – Checks whether the reconstructed document matches the original text.

Paragraph-based chunking preserves the structure of the document, but it does not enforce a maximum chunk size. A long paragraph can therefore produce an oversized chunk.

In [9]:
def paragraph_based_chunking(text):
    print("=" * 70)
    print("STEP 5: PARAGRAPH-BASED CHUNKING")
    print("=" * 70)

    print("\n[1] VALIDATING INPUT")

    if not isinstance(text, str):
        raise TypeError("Input must be a string.")

    text = text.strip()

    if not text:
        print("Input is empty. No chunks will be created.")
        return []

    print("Input validation: PASSED")
    print(f"Total characters: {len(text)}")

    print("\n[2] IDENTIFYING PARAGRAPHS")

    paragraphs = re.split(r"\n\s*\n", text)

    paragraphs = [
        paragraph.strip()
        for paragraph in paragraphs
        if paragraph.strip()
    ]

    print(f"Total paragraphs identified: {len(paragraphs)}")

    print("\n[3] CREATING PARAGRAPH CHUNKS")
    print("-" * 70)

    chunks = []

    for index, paragraph in enumerate(paragraphs, start=1):
        chunks.append(paragraph)

        print(f"\nChunk {index}")
        print(f"Length: {len(paragraph)} characters")
        print(f"Content:\n{paragraph}")

    print("\n[4] VALIDATING RESULTS")
    print("-" * 70)

    print(f"Total chunks created: {len(chunks)}")
    print(f"All chunks are non-empty: {all(chunks)}")

    reconstructed = "\n\n".join(chunks)
    print(
        "Paragraph content preserved:",
        reconstructed == text
    )

    print("\nRESULT: Paragraph-based chunking completed.")

    return chunks

### Executing Paragraph-Based Chunking

This cell applies paragraph-based chunking to the sample document.

- **`paragraph_based_chunking(document)`** – Splits the document using blank-line boundaries.
- **`paragraph_chunks`** – Stores the resulting list for later comparison and retrieval experiments.

The sample document should produce six chunks: three policy headings and three corresponding paragraphs.

The function also checks whether the paragraph content can be reconstructed using blank-line separators.

This cell demonstrates how paragraph boundaries preserve the structure of the original document.

In [10]:
paragraph_chunks = paragraph_based_chunking(document)

STEP 5: PARAGRAPH-BASED CHUNKING

[1] VALIDATING INPUT
Input validation: PASSED
Total characters: 619

[2] IDENTIFYING PARAGRAPHS
Total paragraphs identified: 6

[3] CREATING PARAGRAPH CHUNKS
----------------------------------------------------------------------

Chunk 1
Length: 19 characters
Content:
Annual Leave Policy

Chunk 2
Length: 172 characters
Content:
Employees receive 20 days of annual leave each year.
Leave requests must be approved by their managers.
Employees should submit leave requests five working days in advance.

Chunk 3
Length: 17 characters
Content:
Sick Leave Policy

Chunk 4
Length: 175 characters
Content:
Employees can request sick leave when they are unwell.
A medical certificate may be required for extended absence.
Employees must inform their managers when taking sick leave.

Chunk 5
Length: 21 characters
Content:
Work From Home Policy

Chunk 6
Length: 205 characters
Content:
Eligible employees may work from home with manager approval.
Employees must attend sc

### Installing Libraries for Semantic Chunking

This cell installs the external libraries required for the semantic chunking experiment.

- **`%pip`** – Runs the package installer within the notebook's Python environment.
- **`sentence-transformers`** – Provides pretrained models for converting sentences into numerical embeddings.
- **`scikit-learn`** – Provides machine-learning utilities, including cosine similarity.
- **`-q`** – Reduces the amount of installation output.

These libraries are needed because semantic chunking compares the meaning of sentences using their embeddings.

**Note:** This cell is only necessary when the required packages are not already installed. Keep it as an optional setup cell for reproducibility.

In [11]:
%pip install -q sentence-transformers scikit-learn

Note: you may need to restart the kernel to use updated packages.


### Importing Libraries for Semantic Chunking

This cell imports the classes and functions used to generate sentence embeddings and calculate similarity.

- **`SentenceTransformer`** – Loads a pretrained model that converts text into numerical vectors.
- **`cosine_similarity`** – Calculates the cosine similarity between numerical vectors.
- **`print()`** – Displays confirmation messages showing that the imports succeeded.

These imports prepare the dependencies required by the semantic chunking function.

In [14]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("SentenceTransformer imported successfully.")
print("Cosine similarity imported successfully.")

SentenceTransformer imported successfully.
Cosine similarity imported successfully.


### Understanding the Semantic Embedding Model

This cell displays information about the sentence-embedding model used in the experiment.

- **`print()`** – Displays the experiment heading, model name, and purpose.
- **`all-MiniLM-L6-v2`** – The pretrained sentence-transformer model selected for representing sentences as numerical vectors.

The actual model loading happens in the following code cell. This cell only displays information and does not load the model.

**Recommendation:** This cell is optional. Keep it if you want a separate, clearly labelled model-information section; otherwise, combine its printed messages with the model-loading cell.

In [15]:
print("=" * 70)
print("STEP 6: LOADING THE SEMANTIC EMBEDDING MODEL")
print("=" * 70)

print("\nModel: all-MiniLM-L6-v2")
print("Purpose: Convert sentences into numerical vectors.")
print("These vectors help us compare sentence meanings.")

model = SentenceTransformer("all-MiniLM-L6-v2")

print("\nSUCCESS: Embedding model loaded.")

2026-10-21 18:10:29 | INFO | sentence_transformers.base.model | No device provided, using cpu


STEP 6: LOADING THE SEMANTIC EMBEDDING MODEL

Model: all-MiniLM-L6-v2
Purpose: Convert sentences into numerical vectors.
These vectors help us compare sentence meanings.


2026-10-21 18:10:29 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
2026-10-21 18:10:30 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-10-21 18:10:30 | WARNING | huggingface_hub.utils._http | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-10-21 18:10:30 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json?%2Fsentence-transformers%2Fall-MiniLM-L6-v2%2Fresolve%2Fmain%2Fmodules.json=&etag=%22952a9b81c0bfd99800fabf352f69c7ccd46c5e43%22 "HTTP/1.1 200 OK"
2026-10-21 18:10:30 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\vishakha.more\AppData\Local\Programs\Python\Python314\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\vishakha.more\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
2026-10-21 18:10:30 | INFO | httpx | HTTP Request: HEAD htt

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

2026-10-21 18:11:04 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/chat_template.jinja "HTTP/1.1 404 Not Found"
2026-10-21 18:11:04 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/1_Pooling/config.json "HTTP/1.1 307 Temporary Redirect"
2026-10-21 18:11:04 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/1_Pooling%2Fconfig.json?%2Fsentence-transformers%2Fall-MiniLM-L6-v2%2Fresolve%2Fmain%2F1_Pooling%2Fconfig.json=&etag=%22d1514c3162bbe87b343f565fadc62e6c06f04f03%22 "HTTP/1.1 200 OK"
2026-10-21 18:11:04 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/1_Pooling%2Fconfig.json?%2Fsentence-transformers%2Fa

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

2026-10-21 18:11:05 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/2_Normalize/config.json "HTTP/1.1 404 Not Found"
2026-10-21 18:11:05 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2 "HTTP/1.1 200 OK"



SUCCESS: Embedding model loaded.


### Loading the Sentence-Embedding Model

This cell loads the pretrained model used by the semantic chunking function.

- **`import re`** – Imports regular expression utilities for splitting text.
- **`SentenceTransformer`** – Provides the model-loading interface.
- **`cosine_similarity`** – Provides the function used to compare sentence embeddings.
- **`model = SentenceTransformer("all-MiniLM-L6-v2")`** – Loads the selected model and stores it in the `model` variable.

The model converts sentences into 384-dimensional numerical embeddings. These vectors allow the program to estimate how similar neighboring sentences are in meaning.

The first model-loading attempt may download files and take some time. Subsequent runs can usually use the local cache.

**Important:** Keep one model-loading cell. The repeated imports in this cell are unnecessary if the imports have already been completed successfully.

In [18]:
import re
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Imports successful.")
print("Embedding model is ready.")

2026-10-21 18:25:03 | INFO | sentence_transformers.base.model | No device provided, using cpu
2026-10-21 18:25:03 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-10-21 18:25:03 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json?%2Fsentence-transformers%2Fall-MiniLM-L6-v2%2Fresolve%2Fmain%2Fmodules.json=&etag=%22952a9b81c0bfd99800fabf352f69c7ccd46c5e43%22 "HTTP/1.1 200 OK"
2026-10-21 18:25:03 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-10-21 18:25:03 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/c

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

2026-10-21 18:25:06 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2 "HTTP/1.1 200 OK"
2026-10-21 18:25:06 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-10-21 18:25:07 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-10-21 18:25:07 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/1_Pooling/config.json "HTTP/1.1 307 Temporary Redirect"
2026-10-21 18:25:07 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/1_Pooling%2Fconfig.json?%2Fsentence-transformers%2Fall-MiniLM-L6-v2%2Fresolve%2Fm

Imports successful.
Embedding model is ready.


### Implementing Semantic Chunking

This cell defines a function that groups adjacent sentence-like units using their embedding similarity.

- **`text`** – The document to process.
- **`similarity_threshold=0.45`** – The minimum similarity score used to decide whether adjacent sentence units should remain in the same chunk.
- **Input validation** – Checks the input type, validates the threshold, and handles empty text.
- **`re.split(r'(?<=[.!?])\s+', text)`** – Splits the document into sentence-like units.
- **`model.encode(sentences, normalize_embeddings=True)`** – Converts each sentence into a normalized numerical embedding.
- **`cosine_similarity(...)`** – Calculates similarity between sentence embeddings.
- **Threshold comparison** – Groups adjacent sentences when their similarity meets the configured threshold; otherwise, it starts a new chunk.
- **Validation and print statements** – Display similarity information, resulting chunks, and basic checks.
- **`return chunks`** – Returns the generated semantic chunks.

A lower threshold generally allows more sentences to be grouped together, while a higher threshold generally produces smaller chunks.

**Important limitation:** This implementation compares adjacent sentence embeddings. Similarity scores are heuristic measurements, not probabilities that sentences belong to the same topic. The method may not always produce ideal topic boundaries.

In [20]:
def semantic_chunking(text, similarity_threshold=0.45):
    print("=" * 70)
    print("STEP 6: SEMANTIC CHUNKING")
    print("=" * 70)

    # 1. Validate input
    print("\n[1] VALIDATING INPUT")

    if not isinstance(text, str):
        raise TypeError("Input must be a string.")

    if not 0 <= similarity_threshold <= 1:
        raise ValueError("Threshold must be between 0 and 1.")

    text = text.strip()

    if not text:
        print("Input is empty. No chunks created.")
        return []

    print(f"Total characters: {len(text)}")
    print(f"Similarity threshold: {similarity_threshold}")

    # 2. Split text into sentences
    print("\n[2] SPLITTING TEXT INTO SENTENCES")

    sentences = re.split(r'(?<=[.!?])\s+', text)
    sentences = [s.strip() for s in sentences if s.strip()]

    print(f"Total sentences identified: {len(sentences)}")

    for i, sentence in enumerate(sentences, start=1):
        print(f"Sentence {i}: {sentence}")

    if len(sentences) == 1:
        print("\nOnly one sentence found. Returning one chunk.")
        return sentences

    # 3. Generate embeddings
    print("\n[3] GENERATING SENTENCE EMBEDDINGS")

    embeddings = model.encode(sentences, normalize_embeddings=True)

    print(f"Number of sentence embeddings: {len(embeddings)}")
    print(f"Embedding dimensions: {embeddings.shape[1]}")

    # 4. Compare adjacent sentence meanings
    print("\n[4] CALCULATING SEMANTIC SIMILARITY")

    chunks = []
    current_chunk = [sentences[0]]

    for i in range(1, len(sentences)):
        similarity = cosine_similarity(
            [embeddings[i - 1]],
            [embeddings[i]]
        )[0][0]

        print(f"\nSentence {i} vs Sentence {i + 1}")
        print(f"Similarity score: {similarity:.4f}")

        if similarity >= similarity_threshold:
            print("Decision: Add sentence to the current chunk.")
            current_chunk.append(sentences[i])
        else:
            print("Decision: Start a new chunk.")
            chunks.append(" ".join(current_chunk))
            current_chunk = [sentences[i]]

    # Save the final group
    chunks.append(" ".join(current_chunk))

    # 5. Display final chunks
    print("\n[5] FINAL SEMANTIC CHUNKS")
    print("-" * 70)

    for i, chunk in enumerate(chunks, start=1):
        print(f"\nChunk {i}")
        print(f"Length: {len(chunk)} characters")
        print(f"Content: {chunk}")

    print(f"\nTotal semantic chunks: {len(chunks)}")
    print("Semantic chunking completed successfully.")

    return chunks

### Executing Semantic Chunking

This cell runs the semantic chunking function using the sample document.

- **`document`** – Supplies the input text.
- **`similarity_threshold=0.45`** – Sets the similarity threshold.
- **`semantic_chunking(...)`** – Executes the semantic grouping logic.
- **`semantic_chunks`** – Stores the resulting chunks for comparison and retrieval experiments.

The output displays the resulting groups and the similarity information used to form them.

Review both the number of chunks and their contents. A lower chunk count does not necessarily mean that the semantic grouping is more accurate.

In [21]:
semantic_chunks = semantic_chunking(
    document,
    similarity_threshold=0.45
)

STEP 6: SEMANTIC CHUNKING

[1] VALIDATING INPUT
Total characters: 619
Similarity threshold: 0.45

[2] SPLITTING TEXT INTO SENTENCES
Total sentences identified: 9
Sentence 1: Annual Leave Policy

Employees receive 20 days of annual leave each year.
Sentence 2: Leave requests must be approved by their managers.
Sentence 3: Employees should submit leave requests five working days in advance.
Sentence 4: Sick Leave Policy

Employees can request sick leave when they are unwell.
Sentence 5: A medical certificate may be required for extended absence.
Sentence 6: Employees must inform their managers when taking sick leave.
Sentence 7: Work From Home Policy

Eligible employees may work from home with manager approval.
Sentence 8: Employees must attend scheduled meetings and remain available during working hours.
Sentence 9: Company data must be handled securely when working remotely.

[3] GENERATING SENTENCE EMBEDDINGS


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Number of sentence embeddings: 9
Embedding dimensions: 384

[4] CALCULATING SEMANTIC SIMILARITY

Sentence 1 vs Sentence 2
Similarity score: 0.4358
Decision: Start a new chunk.

Sentence 2 vs Sentence 3
Similarity score: 0.6755
Decision: Add sentence to the current chunk.

Sentence 3 vs Sentence 4
Similarity score: 0.4731
Decision: Add sentence to the current chunk.

Sentence 4 vs Sentence 5
Similarity score: 0.3834
Decision: Start a new chunk.

Sentence 5 vs Sentence 6
Similarity score: 0.3551
Decision: Start a new chunk.

Sentence 6 vs Sentence 7
Similarity score: 0.3131
Decision: Start a new chunk.

Sentence 7 vs Sentence 8
Similarity score: 0.3926
Decision: Start a new chunk.

Sentence 8 vs Sentence 9
Similarity score: 0.2839
Decision: Start a new chunk.

[5] FINAL SEMANTIC CHUNKS
----------------------------------------------------------------------

Chunk 1
Length: 73 characters
Content: Annual Leave Policy

Employees receive 20 days of annual leave each year.

Chunk 2
Length: 193

### Comparing the Implemented Chunking Techniques

This cell creates a pandas DataFrame summarizing the five chunking techniques.

- **`import pandas as pd`** – Imports pandas for tabular data.
- **`pd.DataFrame([...])`** – Creates a table containing the results and characteristics of each technique.
- **`len(fixed_chunks)`** – Counts the chunks produced by fixed-size chunking.
- **`len(overlapping_chunks)`** – Counts the overlapping chunks.
- **`len(sentence_chunks)`** – Counts the sentence-based chunks.
- **`len(paragraph_chunks)`** – Counts the paragraph-based chunks.
- **`len(semantic_chunks)`** – Counts the semantic chunks.
- **`comparison`** – Stores the completed DataFrame.
- **`comparison`** at the end of the cell displays the table in Jupyter Notebook.

The table compares chunk counts and the main characteristics of the implemented methods.

**Why this cell is needed:** It provides a compact summary before testing how chunking parameters affect the results.

In [22]:
import pandas as pd

comparison = pd.DataFrame([
    {
        "Technique": "Fixed-size",
        "Chunks": len(fixed_chunks),
        "Overlap": "No",
        "Main characteristic": "Fixed character limit"
    },
    {
        "Technique": "Overlapping",
        "Chunks": len(overlapping_chunks),
        "Overlap": "40 characters",
        "Main characteristic": "Preserves context at boundaries"
    },
    {
        "Technique": "Sentence-based",
        "Chunks": len(sentence_chunks),
        "Overlap": "No",
        "Main characteristic": "Splits at sentence boundaries"
    },
    {
        "Technique": "Paragraph-based",
        "Chunks": len(paragraph_chunks),
        "Overlap": "No",
        "Main characteristic": "Preserves paragraph structure"
    },
    {
        "Technique": "Semantic",
        "Chunks": len(semantic_chunks),
        "Overlap": "No",
        "Main characteristic": "Groups adjacent sentences by similarity"
    }
])

comparison

,Technique,Chunks,Overlap,Main characteristic
0,Fixed-size,7,No,Fixed character limit
1,Overlapping,10,40 characters,Preserves context at boundaries
2,Sentence-based,9,No,Splits at sentence boundaries
3,Paragraph-based,6,No,Preserves paragraph structure
4,Semantic,7,No,Groups adjacent sentences by similarity


### Chunk Size Comparison

**Objective:** To understand how different chunk sizes affect the number and length of chunks.

This cell tests fixed-size chunking with four different chunk sizes.

- **`chunk_sizes = [50, 100, 200, 300]`** – Defines the chunk sizes to test.
- **`len(document)`** – Displays the original document length.
- **`chunk_size_results = {}`** – Creates a dictionary to store the results for each chunk size.
- **`for size in chunk_sizes`** – Iterates through the selected sizes.
- **`fixed_size_chunking(document, chunk_size=size)`** – Splits the same document using the current size.
- **`chunk_size_results[size] = chunks`** – Stores the resulting chunks using the chunk size as the dictionary key.
- **`sum(len(chunk) for chunk in chunks)`** – Calculates the total number of characters across the chunks.
- **`min()` and `max()`** – Identify the shortest and longest chunk lengths.
- **`enumerate()`** – Displays the length of each chunk.

**What to observe:** Smaller chunk sizes generally produce more chunks. Larger chunk sizes produce fewer chunks but retain more text in each chunk.

This experiment demonstrates the trade-off between chunk granularity and context retention.

In [23]:

# EXPERIMENT 1: CHUNK SIZE COMPARISON

chunk_sizes = [50, 100, 200, 300]

print("=" * 75)
print("EXPERIMENT 1: CHUNK SIZE COMPARISON")
print("=" * 75)

print(f"\nOriginal document length: {len(document)} characters")
print(f"Chunk sizes to test: {chunk_sizes}")

chunk_size_results = {}

for size in chunk_sizes:
    print("\n" + "=" * 75)
    print(f"TESTING CHUNK SIZE: {size} CHARACTERS")
    print("=" * 75)

    chunks = fixed_size_chunking(document, chunk_size=size)
    chunk_size_results[size] = chunks

    print("\nEXPERIMENT SUMMARY")
    print("-" * 50)
    print(f"Selected chunk size: {size}")
    print(f"Number of chunks: {len(chunks)}")
    print(f"Average chunk length: "
          f"{sum(len(chunk) for chunk in chunks) / len(chunks):.2f} characters")
    print(f"Shortest chunk: {min(len(chunk) for chunk in chunks)} characters")
    print(f"Longest chunk: {max(len(chunk) for chunk in chunks)} characters")

    print("\nCHUNK LENGTHS")
    for index, chunk in enumerate(chunks, start=1):
        print(f"Chunk {index}: {len(chunk)} characters")

print("\n" + "=" * 75)
print("ALL CHUNK SIZE EXPERIMENTS COMPLETED")
print("=" * 75)


EXPERIMENT 1: CHUNK SIZE COMPARISON

Original document length: 619 characters
Chunk sizes to test: [50, 100, 200, 300]

TESTING CHUNK SIZE: 50 CHARACTERS
STEP 2: FIXED-SIZE CHUNKING

[1] VALIDATING INPUT
Input validation: PASSED
Total input characters: 619
Selected chunk size: 50 characters
Overlap: 0 characters

[2] CALCULATING CHUNKS
Calculation: 619 / 50
Expected number of chunks: 13
Each chunk contains at most 50 characters.

[3] SPLITTING THE DOCUMENT
----------------------------------------------------------------------

Chunk 1
Character positions: 0 to 49
Chunk length: 50 characters
Chunk content: 'Annual Leave Policy\n\nEmployees receive 20 days of '

Chunk 2
Character positions: 50 to 99
Chunk length: 50 characters
Chunk content: 'annual leave each year.\nLeave requests must be app'

Chunk 3
Character positions: 100 to 149
Chunk length: 50 characters
Chunk content: 'roved by their managers.\nEmployees should submit l'

Chunk 4
Character positions: 150 to 199
Chunk length: 50 

### Final Comparison: Chunk Size vs. Chunk Count

This cell summarizes the results collected in Experiment 1.

- **`chunk_size_results.items()`** – Iterates through each chunk size and its corresponding chunks.
- **`lengths`** – Stores the length of every chunk.
- **`average_length`** – Calculates the average chunk length.
- **`len(chunks)`** – Reports the number of chunks.
- **`min(lengths)` and `max(lengths)`** – Identify the minimum and maximum chunk lengths.
- **Formatted f-strings** – Align the output into readable columns.

**What to observe:** Increasing the chunk size should generally reduce the number of chunks while increasing their average length.

This summary makes it easier to compare the tested configurations without reviewing every chunk individually.

In [24]:

print("=" * 75)
print("FINAL COMPARISON: CHUNK SIZE vs CHUNK COUNT")
print("=" * 75)

print(f"{'Chunk Size':<15}{'Number of Chunks':<20}"
      f"{'Average Length':<20}{'Length Range'}")
print("-" * 75)

for size, chunks in chunk_size_results.items():
    lengths = [len(chunk) for chunk in chunks]
    average_length = sum(lengths) / len(lengths)

    print(
        f"{size:<15}"
        f"{len(chunks):<20}"
        f"{average_length:<20.2f}"
        f"{min(lengths)}-{max(lengths)} characters"
    )


FINAL COMPARISON: CHUNK SIZE vs CHUNK COUNT
Chunk Size     Number of Chunks    Average Length      Length Range
---------------------------------------------------------------------------
50             13                  47.62               19-50 characters
100            7                   88.43               19-100 characters
200            4                   154.75              19-200 characters
300            3                   206.33              19-300 characters


### Overlap Comparison

**Objective:** To understand how overlap affects chunk count, chunk length, and repeated text.

This cell keeps the chunk size fixed at 100 characters and tests four overlap values.

- **`chunk_size = 100`** – Sets the maximum chunk length.
- **`overlap_values = [0, 20, 40, 60]`** – Defines the overlap values to test.
- **`overlap_results = {}`** – Stores the generated chunks for each overlap value.
- **`overlapping_chunking(...)`** – Creates chunks using the current overlap.
- **`chunk_size - overlap`** – Calculates the step size.
- **`lengths`** – Stores each chunk's length.
- **`average_length`** – Calculates the average chunk length.
- **Boundary comparison loop** – Checks the suffix of one chunk against the prefix of the next to find their shared characters.

**What to observe:** Increasing overlap generally creates more chunks and increases the total amount of repeated text. An overlap of zero produces non-overlapping chunks.

**Why this experiment is useful:** It helps demonstrate how overlap preserves context while introducing additional storage and retrieval redundancy.

In [25]:

# OVERLAP COMPARISON

chunk_size = 100
overlap_values = [0, 20, 40, 60]

overlap_results = {}

print("=" * 75)
print("EXPERIMENT 2: OVERLAP COMPARISON")
print("=" * 75)

print(f"\nDocument length: {len(document)} characters")
print(f"Fixed chunk size: {chunk_size} characters")
print(f"Overlap values to test: {overlap_values}")

for overlap in overlap_values:
    print("\n" + "=" * 75)
    print(f"TESTING OVERLAP: {overlap} CHARACTERS")
    print("=" * 75)

    chunks = overlapping_chunking(
        document,
        chunk_size=chunk_size,
        overlap=overlap
    )

    overlap_results[overlap] = chunks

    lengths = [len(chunk) for chunk in chunks]
    average_length = sum(lengths) / len(lengths)

    print("\nEXPERIMENT SUMMARY")
    print("-" * 50)
    print(f"Chunk size: {chunk_size}")
    print(f"Overlap: {overlap}")
    print(f"Step size: {chunk_size - overlap}")
    print(f"Number of chunks: {len(chunks)}")
    print(f"Average chunk length: {average_length:.2f} characters")
    print(f"Shortest chunk: {min(lengths)} characters")
    print(f"Longest chunk: {max(lengths)} characters")

    if overlap > 0 and len(chunks) > 1:
        shared_text = sum(
            len(set()) for _ in []
        )
        print("\nChecking consecutive chunk boundaries:")

        for i in range(len(chunks) - 1):
            actual_shared = 0
            max_possible = min(len(chunks[i]), len(chunks[i + 1]))

            for size in range(1, max_possible + 1):
                if chunks[i][-size:] == chunks[i + 1][:size]:
                    actual_shared = size

            print(
                f"Chunks {i + 1} and {i + 2}: "
                f"{actual_shared} shared characters"
            )

print("\n" + "=" * 75)
print("OVERLAP COMPARISON COMPLETED")
print("=" * 75)


EXPERIMENT 2: OVERLAP COMPARISON

Document length: 619 characters
Fixed chunk size: 100 characters
Overlap values to test: [0, 20, 40, 60]

TESTING OVERLAP: 0 CHARACTERS
STEP 3: OVERLAPPING CHUNKING

[1] VALIDATING INPUT
Input validation: PASSED
Total characters: 619
Chunk size: 100
Overlap: 0

[2] CALCULATING STEP SIZE
Step size = 100 - 0 = 100

[3] CREATING OVERLAPPING CHUNKS
----------------------------------------------------------------------

Chunk 1
Character positions: 0 to 99
Chunk length: 100 characters
Chunk content: 'Annual Leave Policy\n\nEmployees receive 20 days of annual leave each year.\nLeave requests must be app'

Chunk 2
Character positions: 100 to 199
Chunk length: 100 characters
Chunk content: 'roved by their managers.\nEmployees should submit leave requests five working days in advance.\n\nSick '

Chunk 3
Character positions: 200 to 299
Chunk length: 100 characters
Chunk content: 'Leave Policy\n\nEmployees can request sick leave when they are unwell.\nA medical c

### Final Comparison: Overlap vs. Chunk Count

This cell summarizes the results of Experiment 2.

- **`overlap_results.items()`** – Iterates through each overlap value and its chunks.
- **`lengths`** – Stores the length of each chunk.
- **`average_length`** – Calculates the average chunk length.
- **`sum(lengths)`** – Calculates the total number of characters across all chunks, including repeated overlap.
- **`chunk_size - overlap`** – Calculates the step size for each overlap value.
- **Formatted f-strings** – Display the results in aligned columns.

**What to observe:** The total number of characters across all chunks can exceed the original document length because some text is repeated between consecutive chunks.

This comparison demonstrates the trade-off between preserving context and increasing redundancy.

In [26]:

print("=" * 75)
print("FINAL COMPARISON: OVERLAP vs CHUNK COUNT")
print("=" * 75)

print(
    f"{'Overlap':<12}"
    f"{'Step Size':<12}"
    f"{'Chunk Count':<15}"
    f"{'Average Length':<18}"
    f"{'Total Characters in Chunks'}"
)
print("-" * 75)

for overlap, chunks in overlap_results.items():
    lengths = [len(chunk) for chunk in chunks]
    average_length = sum(lengths) / len(lengths)
    total_chunk_characters = sum(lengths)

    print(
        f"{overlap:<12}"
        f"{chunk_size - overlap:<12}"
        f"{len(chunks):<15}"
        f"{average_length:<18.2f}"
        f"{total_chunk_characters}"
    )


FINAL COMPARISON: OVERLAP vs CHUNK COUNT
Overlap     Step Size   Chunk Count    Average Length    Total Characters in Chunks
---------------------------------------------------------------------------
0           100         7              88.43             619
20          80          8              94.88             759
40          60          10             97.90             979
60          40          14             99.93             1399


### Semantic Similarity Threshold Comparison

**Objective:** To understand how the similarity threshold affects semantic chunk formation.

This cell tests semantic chunking with four different similarity thresholds.

- **`similarity_thresholds = [0.30, 0.45, 0.60, 0.75]`** – Defines the thresholds to test.
- **`semantic_threshold_results = {}`** – Stores the chunks generated for each threshold.
- **`for threshold in similarity_thresholds`** – Iterates through the selected thresholds.
- **`semantic_chunking(...)`** – Processes the document using the current threshold.
- **`semantic_threshold_results[threshold] = chunks`** – Stores the resulting chunks.
- **`lengths`** – Records each chunk's length.
- **`sum()`, `min()`, and `max()`** – Calculate the average, shortest, and longest chunk lengths.
- **`enumerate()`** – Displays each chunk and its length.

**What to observe:** A lower threshold generally groups more adjacent sentence units together. A higher threshold generally creates more, smaller chunks.

The results depend on the model's similarity scores and the sample document. A threshold is a heuristic, not a universal measure of chunk quality.

In [27]:

# SEMANTIC THRESHOLD COMPARISON

similarity_thresholds = [0.30, 0.45, 0.60, 0.75]

semantic_threshold_results = {}

print("=" * 75)
print("EXPERIMENT 3: SEMANTIC THRESHOLD COMPARISON")
print("=" * 75)

print(f"\nDocument length: {len(document)} characters")
print(f"Embedding model: all-MiniLM-L6-v2")
print(f"Thresholds to test: {similarity_thresholds}")

for threshold in similarity_thresholds:
    print("\n" + "=" * 75)
    print(f"TESTING SIMILARITY THRESHOLD: {threshold}")
    print("=" * 75)

    chunks = semantic_chunking(
        document,
        similarity_threshold=threshold
    )

    semantic_threshold_results[threshold] = chunks

    lengths = [len(chunk) for chunk in chunks]

    print("\nEXPERIMENT SUMMARY")
    print("-" * 50)
    print(f"Similarity threshold: {threshold}")
    print(f"Number of chunks: {len(chunks)}")
    print(
        f"Average chunk length: "
        f"{sum(lengths) / len(lengths):.2f} characters"
    )
    print(f"Shortest chunk: {min(lengths)} characters")
    print(f"Longest chunk: {max(lengths)} characters")

    print("\nCHUNK GROUPING")
    for index, chunk in enumerate(chunks, start=1):
        print(f"\nChunk {index} ({len(chunk)} characters)")
        print(chunk)

print("\n" + "=" * 75)
print("SEMANTIC THRESHOLD EXPERIMENTS COMPLETED")
print("=" * 75)


EXPERIMENT 3: SEMANTIC THRESHOLD COMPARISON

Document length: 619 characters
Embedding model: all-MiniLM-L6-v2
Thresholds to test: [0.3, 0.45, 0.6, 0.75]

TESTING SIMILARITY THRESHOLD: 0.3
STEP 6: SEMANTIC CHUNKING

[1] VALIDATING INPUT
Total characters: 619
Similarity threshold: 0.3

[2] SPLITTING TEXT INTO SENTENCES
Total sentences identified: 9
Sentence 1: Annual Leave Policy

Employees receive 20 days of annual leave each year.
Sentence 2: Leave requests must be approved by their managers.
Sentence 3: Employees should submit leave requests five working days in advance.
Sentence 4: Sick Leave Policy

Employees can request sick leave when they are unwell.
Sentence 5: A medical certificate may be required for extended absence.
Sentence 6: Employees must inform their managers when taking sick leave.
Sentence 7: Work From Home Policy

Eligible employees may work from home with manager approval.
Sentence 8: Employees must attend scheduled meetings and remain available during working hour

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Number of sentence embeddings: 9
Embedding dimensions: 384

[4] CALCULATING SEMANTIC SIMILARITY

Sentence 1 vs Sentence 2
Similarity score: 0.4358
Decision: Add sentence to the current chunk.

Sentence 2 vs Sentence 3
Similarity score: 0.6755
Decision: Add sentence to the current chunk.

Sentence 3 vs Sentence 4
Similarity score: 0.4731
Decision: Add sentence to the current chunk.

Sentence 4 vs Sentence 5
Similarity score: 0.3834
Decision: Add sentence to the current chunk.

Sentence 5 vs Sentence 6
Similarity score: 0.3551
Decision: Add sentence to the current chunk.

Sentence 6 vs Sentence 7
Similarity score: 0.3131
Decision: Add sentence to the current chunk.

Sentence 7 vs Sentence 8
Similarity score: 0.3926
Decision: Add sentence to the current chunk.

Sentence 8 vs Sentence 9
Similarity score: 0.2839
Decision: Start a new chunk.

[5] FINAL SEMANTIC CHUNKS
----------------------------------------------------------------------

Chunk 1
Length: 556 characters
Content: Annual Leave 

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Number of sentence embeddings: 9
Embedding dimensions: 384

[4] CALCULATING SEMANTIC SIMILARITY

Sentence 1 vs Sentence 2
Similarity score: 0.4358
Decision: Start a new chunk.

Sentence 2 vs Sentence 3
Similarity score: 0.6755
Decision: Add sentence to the current chunk.

Sentence 3 vs Sentence 4
Similarity score: 0.4731
Decision: Add sentence to the current chunk.

Sentence 4 vs Sentence 5
Similarity score: 0.3834
Decision: Start a new chunk.

Sentence 5 vs Sentence 6
Similarity score: 0.3551
Decision: Start a new chunk.

Sentence 6 vs Sentence 7
Similarity score: 0.3131
Decision: Start a new chunk.

Sentence 7 vs Sentence 8
Similarity score: 0.3926
Decision: Start a new chunk.

Sentence 8 vs Sentence 9
Similarity score: 0.2839
Decision: Start a new chunk.

[5] FINAL SEMANTIC CHUNKS
----------------------------------------------------------------------

Chunk 1
Length: 73 characters
Content: Annual Leave Policy

Employees receive 20 days of annual leave each year.

Chunk 2
Length: 193

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Number of sentence embeddings: 9
Embedding dimensions: 384

[4] CALCULATING SEMANTIC SIMILARITY

Sentence 1 vs Sentence 2
Similarity score: 0.4358
Decision: Start a new chunk.

Sentence 2 vs Sentence 3
Similarity score: 0.6755
Decision: Add sentence to the current chunk.

Sentence 3 vs Sentence 4
Similarity score: 0.4731
Decision: Start a new chunk.

Sentence 4 vs Sentence 5
Similarity score: 0.3834
Decision: Start a new chunk.

Sentence 5 vs Sentence 6
Similarity score: 0.3551
Decision: Start a new chunk.

Sentence 6 vs Sentence 7
Similarity score: 0.3131
Decision: Start a new chunk.

Sentence 7 vs Sentence 8
Similarity score: 0.3926
Decision: Start a new chunk.

Sentence 8 vs Sentence 9
Similarity score: 0.2839
Decision: Start a new chunk.

[5] FINAL SEMANTIC CHUNKS
----------------------------------------------------------------------

Chunk 1
Length: 73 characters
Content: Annual Leave Policy

Employees receive 20 days of annual leave each year.

Chunk 2
Length: 119 characters
Cont

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Number of sentence embeddings: 9
Embedding dimensions: 384

[4] CALCULATING SEMANTIC SIMILARITY

Sentence 1 vs Sentence 2
Similarity score: 0.4358
Decision: Start a new chunk.

Sentence 2 vs Sentence 3
Similarity score: 0.6755
Decision: Start a new chunk.

Sentence 3 vs Sentence 4
Similarity score: 0.4731
Decision: Start a new chunk.

Sentence 4 vs Sentence 5
Similarity score: 0.3834
Decision: Start a new chunk.

Sentence 5 vs Sentence 6
Similarity score: 0.3551
Decision: Start a new chunk.

Sentence 6 vs Sentence 7
Similarity score: 0.3131
Decision: Start a new chunk.

Sentence 7 vs Sentence 8
Similarity score: 0.3926
Decision: Start a new chunk.

Sentence 8 vs Sentence 9
Similarity score: 0.2839
Decision: Start a new chunk.

[5] FINAL SEMANTIC CHUNKS
----------------------------------------------------------------------

Chunk 1
Length: 73 characters
Content: Annual Leave Policy

Employees receive 20 days of annual leave each year.

Chunk 2
Length: 50 characters
Content: Leave reques

### Final Comparison: Similarity Threshold vs. Chunk Count

This cell summarizes the semantic chunking results from Experiment 3.

- **`semantic_threshold_results.items()`** – Iterates through each threshold and its chunks.
- **`lengths`** – Stores the lengths of all chunks for the current threshold.
- **`len(chunks)`** – Reports the number of chunks.
- **`sum(lengths) / len(lengths)`** – Calculates the average chunk length.
- **`min(lengths)` and `max(lengths)`** – Identify the shortest and longest chunks.
- **Formatted f-strings** – Align the comparison into readable columns.

**What to observe:** Compare how the chunk count and average length change as the threshold increases.

These results describe the behavior of the current model and document. They do not establish that a particular threshold is best for all documents.

In [28]:

print("=" * 75)
print("FINAL COMPARISON: SIMILARITY THRESHOLD vs CHUNK COUNT")
print("=" * 75)

print(
    f"{'Threshold':<15}"
    f"{'Chunk Count':<15}"
    f"{'Average Length':<20}"
    f"{'Shortest Chunk':<18}"
    f"{'Longest Chunk'}"
)
print("-" * 75)

for threshold, chunks in semantic_threshold_results.items():
    lengths = [len(chunk) for chunk in chunks]

    print(
        f"{threshold:<15.2f}"
        f"{len(chunks):<15}"
        f"{sum(lengths) / len(lengths):<20.2f}"
        f"{min(lengths):<18}"
        f"{max(lengths)}"
    )


FINAL COMPARISON: SIMILARITY THRESHOLD vs CHUNK COUNT
Threshold      Chunk Count    Average Length      Shortest Chunk    Longest Chunk
---------------------------------------------------------------------------
0.30           2              308.00              60                556
0.45           7              87.29               59                193
0.60           8              76.25               59                119
0.75           9              67.67               50                83


### Query-Based Retrieval Simulation

**Objective:** To understand how document chunks can be retrieved in response to questions.

This cell defines six questions based on the sample policy document.

- **`queries`** – Stores the test questions and their associated search keywords.
- **`question`** – Contains the natural-language question.
- **`keywords`** – Contains the terms used to score candidate chunks.
- **`enumerate(queries, start=1)`** – Displays each question with its number.
- **`print()`** – Displays the question and its configured keywords.

The queries cover annual leave, leave approvals, advance notice, sick leave, medical certificates, and remote-working requirements.

Using the same queries across all chunking techniques makes their retrieval results easier to compare.

In [29]:

# EXPERIMENT 4: QUERY-BASED RETRIEVAL SIMULATION

queries = [
    {
        "question": "How many annual leave days do employees receive?",
        "keywords": ["annual leave", "20 days"]
    },
    {
        "question": "Who must approve leave requests?",
        "keywords": ["leave requests", "approved", "managers"]
    },
    {
        "question": "How many working days in advance should leave be requested?",
        "keywords": ["five working days", "advance"]
    },
    {
        "question": "When may employees request sick leave?",
        "keywords": ["sick leave", "unwell"]
    },
    {
        "question": "What is required for an extended absence?",
        "keywords": ["medical certificate", "extended absence"]
    },
    {
        "question": "What must employees do when working remotely?",
        "keywords": ["working remotely", "securely"]
    }
]

print("=" * 75)
print("TEST QUESTIONS")
print("=" * 75)

for index, query in enumerate(queries, start=1):
    print(f"\nQuestion {index}: {query['question']}")
    print(f"Search keywords: {query['keywords']}")


TEST QUESTIONS

Question 1: How many annual leave days do employees receive?
Search keywords: ['annual leave', '20 days']

Question 2: Who must approve leave requests?
Search keywords: ['leave requests', 'approved', 'managers']

Question 3: How many working days in advance should leave be requested?
Search keywords: ['five working days', 'advance']

Question 4: When may employees request sick leave?
Search keywords: ['sick leave', 'unwell']

Question 5: What is required for an extended absence?
Search keywords: ['medical certificate', 'extended absence']

Question 6: What must employees do when working remotely?
Search keywords: ['working remotely', 'securely']


### Implementing Keyword-Based Chunk Retrieval

This cell defines `retrieve_chunks()`, a function that ranks document chunks according to the presence of configured query keywords.

- **`query`** – Contains the question and its search keywords.
- **`chunks`** – Contains the candidate chunks to search.
- **`top_k=3`** – Sets the maximum number of results to return.
- **Input validation** – Checks that the query is a dictionary, chunks are supplied as a list, and `top_k` is positive.
- **`keywords = query["keywords"]`** – Retrieves the configured keywords.
- **`chunk.lower()`** – Converts chunk text to lowercase for case-insensitive matching.
- **`matched_keywords`** – Collects the keywords found in each chunk.
- **`len(matched_keywords) / len(keywords)`** – Calculates the proportion of configured keywords found.
- **`scored_chunks.append(...)`** – Stores the chunk ID, content, score, and matched keywords.
- **`sort()`** – Ranks chunks by descending score and then by their original order.
- **`return scored_chunks[:top_k]`** – Returns up to the requested number of highest-ranked chunks.

**Important limitation:** This is a keyword-based retrieval simulation, not embedding-based semantic search. A high score means the configured keywords were found; it does not guarantee that the chunk fully answers the question.

In [30]:

def retrieve_chunks(query, chunks, top_k=3):
    """
    Retrieve chunks using a simple keyword-matching score.

    This is a basic retrieval simulation, not semantic search.
    """
    if not isinstance(query, dict):
        raise TypeError("Query must be a dictionary.")

    if not isinstance(chunks, list):
        raise TypeError("Chunks must be provided as a list.")

    if top_k <= 0:
        raise ValueError("top_k must be greater than zero.")

    keywords = query["keywords"]
    scored_chunks = []

    for index, chunk in enumerate(chunks):
        chunk_lower = chunk.lower()

        # Count how many query keywords appear in this chunk.
        matched_keywords = [
            keyword
            for keyword in keywords
            if keyword.lower() in chunk_lower
        ]

        score = len(matched_keywords) / len(keywords)

        if score > 0:
            scored_chunks.append({
                "chunk_id": index + 1,
                "chunk": chunk,
                "score": score,
                "matched_keywords": matched_keywords
            })

    # Sort by score, then by original chunk order.
    scored_chunks.sort(
        key=lambda item: (-item["score"], item["chunk_id"])
    )

    return scored_chunks[:top_k]


### Running Retrieval with Sentence-Based Chunks

This cell uses the sentence-based chunks to test the retrieval function.

- **`retrieval_chunks = sentence_chunks`** – Selects the sentence-based chunks for the first test.
- **`for query in queries`** – Processes each test question.
- **`retrieve_chunks(...)`** – Searches the selected chunks and returns the highest-scoring results.
- **`if not results`** – Handles questions for which no chunks match the keywords.
- **`enumerate(results, start=1)`** – Assigns a rank to each retrieved result.
- **`result['chunk_id']`** – Displays the original chunk number.
- **`result['score']`** – Displays the keyword match score.
- **`result['matched_keywords']`** – Shows which keywords were found.
- **`result['chunk']`** – Displays the retrieved text.

**What to observe:** Check whether the retrieved chunks contain the information needed to answer each question. A score of `1.00` indicates that all configured keywords were found, not that the answer is necessarily complete.

In [31]:

# Use the existing sentence-based chunks for the first test.
retrieval_chunks = sentence_chunks

print("=" * 75)
print("QUERY-BASED RETRIEVAL RESULTS")
print("=" * 75)

for query in queries:
    print("\n" + "=" * 75)
    print(f"QUESTION: {query['question']}")
    print("=" * 75)

    results = retrieve_chunks(
        query=query,
        chunks=retrieval_chunks,
        top_k=3
    )

    if not results:
        print("No chunks matched the query keywords.")
        continue

    print(f"Retrieved chunks: {len(results)}")

    for rank, result in enumerate(results, start=1):
        print(f"\nRank: {rank}")
        print(f"Chunk ID: {result['chunk_id']}")
        print(f"Keyword match score: {result['score']:.2f}")
        print(f"Matched keywords: {result['matched_keywords']}")
        print(f"Chunk content:\n{result['chunk']}")


QUERY-BASED RETRIEVAL RESULTS

QUESTION: How many annual leave days do employees receive?
Retrieved chunks: 1

Rank: 1
Chunk ID: 1
Keyword match score: 1.00
Matched keywords: ['annual leave', '20 days']
Chunk content:
Annual Leave Policy

Employees receive 20 days of annual leave each year.

QUESTION: Who must approve leave requests?
Retrieved chunks: 3

Rank: 1
Chunk ID: 2
Keyword match score: 1.00
Matched keywords: ['leave requests', 'approved', 'managers']
Chunk content:
Leave requests must be approved by their managers.

Rank: 2
Chunk ID: 3
Keyword match score: 0.33
Matched keywords: ['leave requests']
Chunk content:
Employees should submit leave requests five working days in advance.

Rank: 3
Chunk ID: 6
Keyword match score: 0.33
Matched keywords: ['managers']
Chunk content:
Employees must inform their managers when taking sick leave.

QUESTION: How many working days in advance should leave be requested?
Retrieved chunks: 1

Rank: 1
Chunk ID: 3
Keyword match score: 1.00
Matched ke

### Comparing Retrieval Across Chunking Techniques

This cell runs the keyword-based retrieval simulation against all five chunking techniques.

- **`chunking_methods`** – Maps each method name to its corresponding list of chunks.
- **`fixed_size_chunking(...)`** – Generates fixed-size chunks with a size of 100 characters.
- **`overlapping_chunking(...)`** – Generates overlapping chunks with a size of 100 and an overlap of 40.
- **`sentence_chunks`** – Reuses the sentence-based chunks.
- **`paragraph_chunks`** – Reuses the paragraph-based chunks.
- **`semantic_chunking(...)`** – Generates semantic chunks using a threshold of 0.45.
- **`for query in queries`** – Tests every question.
- **`for method_name, chunks in chunking_methods.items()`** – Tests every chunking method against the current question.
- **`retrieve_chunks(...)`** – Returns the highest-scoring matching chunks.
- **`print()`** – Displays the method name, chunk ID, score, and matched keywords.

**What to observe:** Fixed-size chunking can split related information across boundaries, while other methods may keep it together. The results depend on the document, chunk boundaries, and keyword matching rules.

This experiment compares retrieval behavior, but it does not evaluate a complete RAG system with embeddings, a vector database, reranking, or an LLM.

In [32]:

# Compare retrieval results using different chunking techniques.

chunking_methods = {
    "Fixed-size (100)": fixed_size_chunking(document, chunk_size=100),
    "Overlapping (100/40)": overlapping_chunking(
        document, chunk_size=100, overlap=40
    ),
    "Sentence-based": sentence_chunks,
    "Paragraph-based": paragraph_chunks,
    "Semantic (0.45)": semantic_chunking(
        document, similarity_threshold=0.45
    )
}

print("\n" + "=" * 85)
print("RETRIEVAL COMPARISON ACROSS CHUNKING TECHNIQUES")
print("=" * 85)

for query in queries:
    print(f"\nQuestion: {query['question']}")

    for method_name, chunks in chunking_methods.items():
        results = retrieve_chunks(query, chunks, top_k=3)

        print(f"\n{method_name}")
        if results:
            for result in results:
                print(
                    f"  Chunk {result['chunk_id']} | "
                    f"Score: {result['score']:.2f} | "
                    f"Matched: {result['matched_keywords']}"
                )
        else:
            print("  No matching chunks found.")


STEP 2: FIXED-SIZE CHUNKING

[1] VALIDATING INPUT
Input validation: PASSED
Total input characters: 619
Selected chunk size: 100 characters
Overlap: 0 characters

[2] CALCULATING CHUNKS
Calculation: 619 / 100
Expected number of chunks: 7
Each chunk contains at most 100 characters.

[3] SPLITTING THE DOCUMENT
----------------------------------------------------------------------

Chunk 1
Character positions: 0 to 99
Chunk length: 100 characters
Chunk content: 'Annual Leave Policy\n\nEmployees receive 20 days of annual leave each year.\nLeave requests must be app'

Chunk 2
Character positions: 100 to 199
Chunk length: 100 characters
Chunk content: 'roved by their managers.\nEmployees should submit leave requests five working days in advance.\n\nSick '

Chunk 3
Character positions: 200 to 299
Chunk length: 100 characters
Chunk content: 'Leave Policy\n\nEmployees can request sick leave when they are unwell.\nA medical certificate may be re'

Chunk 4
Character positions: 300 to 399
Chunk len

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Number of sentence embeddings: 9
Embedding dimensions: 384

[4] CALCULATING SEMANTIC SIMILARITY

Sentence 1 vs Sentence 2
Similarity score: 0.4358
Decision: Start a new chunk.

Sentence 2 vs Sentence 3
Similarity score: 0.6755
Decision: Add sentence to the current chunk.

Sentence 3 vs Sentence 4
Similarity score: 0.4731
Decision: Add sentence to the current chunk.

Sentence 4 vs Sentence 5
Similarity score: 0.3834
Decision: Start a new chunk.

Sentence 5 vs Sentence 6
Similarity score: 0.3551
Decision: Start a new chunk.

Sentence 6 vs Sentence 7
Similarity score: 0.3131
Decision: Start a new chunk.

Sentence 7 vs Sentence 8
Similarity score: 0.3926
Decision: Start a new chunk.

Sentence 8 vs Sentence 9
Similarity score: 0.2839
Decision: Start a new chunk.

[5] FINAL SEMANTIC CHUNKS
----------------------------------------------------------------------

Chunk 1
Length: 73 characters
Content: Annual Leave Policy

Employees receive 20 days of annual leave each year.

Chunk 2
Length: 193

### Preparing the Input for Recursive Splitting

This cell defines the sample policy document used in the recursive splitting experiment.

- **`document`** – Stores the multi-line policy text.
- **`.strip()`** – Removes leading and trailing whitespace.
- **`len(document)`** – Displays the document length.
- **`print(document)`** – Displays the input before recursive splitting.

The document is the same sample used by the earlier chunking techniques.

**Note:** The input document has already been defined in the preparation section. Reusing that variable avoids unnecessary duplication and keeps the notebook easier to maintain.

In [2]:

document = """
Annual Leave Policy

Employees receive 20 days of annual leave each year.
Leave requests must be approved by their managers.
Employees should submit leave requests five working days in advance.

Sick Leave Policy

Employees can request sick leave when they are unwell.
A medical certificate may be required for extended absence.
Employees must inform their managers when taking sick leave.

Work From Home Policy

Eligible employees may work from home with manager approval.
Employees must attend scheduled meetings and remain available during working hours.
Company data must be handled securely when working remotely.
""".strip()

print("Document loaded successfully!")
print(f"Document length: {len(document)} characters")
print(document)


Document loaded successfully!
Document length: 619 characters
Annual Leave Policy

Employees receive 20 days of annual leave each year.
Leave requests must be approved by their managers.
Employees should submit leave requests five working days in advance.

Sick Leave Policy

Employees can request sick leave when they are unwell.
A medical certificate may be required for extended absence.
Employees must inform their managers when taking sick leave.

Work From Home Policy

Eligible employees may work from home with manager approval.
Employees must attend scheduled meetings and remain available during working hours.
Company data must be handled securely when working remotely.


### Implementing Recursive Splitting

**Objective:** To understand how recursive splitting divides a document using a hierarchy of separators while respecting a maximum chunk size.

- **`recursive_chunking(text, chunk_size=100)`** – Defines the recursive chunking function.
- **Input validation** – Checks the input type and ensures the chunk size is a positive integer.
- **`text.strip()`** – Removes leading and trailing whitespace.
- **`separators`** – Defines the order of splitting boundaries: blank lines, line breaks, sentence-like boundaries, spaces, and individual characters.
- **`split_recursively(text_part, separator_index)`** – Recursively splits text that is too large for the configured chunk size.
- **`len(text_part) <= chunk_size`** – Returns the text as one chunk when it already fits.
- **`text_part.split(separator)`** – Divides the text using the current separator.
- **`separator_index + 1`** – Moves to a smaller separator when the current boundary is insufficient.
- **`current`** – Temporarily holds text being combined into a chunk.
- **`candidate`** – Represents the current text combined with the next piece.
- **`len(candidate) <= chunk_size`** – Checks whether the next piece can be added without exceeding the size limit.
- **`chunks.extend(...)`** – Adds the results of recursive calls to the output list.
- **Character-slicing fallback** – Splits text into fixed-size slices when no more separators are available.
- **`all(...)`** – Checks whether every chunk is non-empty and within the maximum size.
- **`return chunks`** – Returns the generated chunks.

The function prioritizes larger structural boundaries before using smaller boundaries. This can help preserve paragraphs, sentences, and words whenever possible.

**Important limitation:** The current implementation may alter or discard whitespace because it splits and reconstructs text using separators. Passing the chunk-size checks does not prove exact text preservation.

In [3]:

# RECURSIVE SPLITTING

def recursive_chunking(text, chunk_size=100):
    print("=" * 70)
    print("EXPERIMENT 6: RECURSIVE SPLITTING")
    print("=" * 70)

    if not isinstance(text, str):
        raise TypeError("Input must be a string.")

    if not isinstance(chunk_size, int) or isinstance(chunk_size, bool):
        raise TypeError("Chunk size must be an integer.")

    if chunk_size <= 0:
        raise ValueError("Chunk size must be greater than zero.")

    text = text.strip()

    if not text:
        print("Input is empty. No chunks will be created.")
        return []

    # Try larger structural boundaries before smaller ones.
    separators = ["\n\n", "\n", ". ", " ", ""]

    print(f"\nDocument length: {len(text)} characters")
    print(f"Maximum chunk size: {chunk_size} characters")
    print(f"Separator hierarchy: {separators}")

    def split_recursively(text_part, separator_index):
        if len(text_part) <= chunk_size:
            return [text_part]

        if separator_index >= len(separators):
            # Final fallback: split by character count.
            return [
                text_part[i:i + chunk_size]
                for i in range(0, len(text_part), chunk_size)
            ]

        separator = separators[separator_index]

        if separator == "":
            pieces = list(text_part)
        else:
            pieces = text_part.split(separator)

        # If this separator cannot divide the text, try the next.
        if len(pieces) <= 1:
            return split_recursively(text_part, separator_index + 1)

        chunks = []
        current = ""

        for piece in pieces:
            candidate = piece if not current else (
                current + separator + piece
            )

            if len(candidate) <= chunk_size:
                current = candidate
            else:
                if current:
                    chunks.extend(
                        split_recursively(current, separator_index + 1)
                    )

                if len(piece) <= chunk_size:
                    current = piece
                else:
                    chunks.extend(
                        split_recursively(piece, separator_index + 1)
                    )
                    current = ""

        if current:
            chunks.extend(
                split_recursively(current, separator_index + 1)
            )

        return chunks

    chunks = split_recursively(text, 0)

    print("\nFINAL CHUNKS")
    print("-" * 70)

    for index, chunk in enumerate(chunks, start=1):
        print(f"\nChunk {index}")
        print(f"Length: {len(chunk)} characters")
        print(f"Content: {chunk!r}")

    print("\nVALIDATION")
    print("-" * 70)
    print(f"Total chunks: {len(chunks)}")
    print(
        "All chunks within size limit:",
        all(0 < len(chunk) <= chunk_size for chunk in chunks)
    )
    print(f"All chunks non-empty: {all(chunks)}")

    print(
        "\nNote: This basic implementation prioritizes separator "
        "boundaries. It does not guarantee exact whitespace preservation."
    )

    return chunks


recursive_chunks = recursive_chunking(document, chunk_size=100)


EXPERIMENT 6: RECURSIVE SPLITTING

Document length: 619 characters
Maximum chunk size: 100 characters
Separator hierarchy: ['\n\n', '\n', '. ', ' ', '']

FINAL CHUNKS
----------------------------------------------------------------------

Chunk 1
Length: 19 characters
Content: 'Annual Leave Policy'

Chunk 2
Length: 52 characters
Content: 'Employees receive 20 days of annual leave each year.'

Chunk 3
Length: 50 characters
Content: 'Leave requests must be approved by their managers.'

Chunk 4
Length: 68 characters
Content: 'Employees should submit leave requests five working days in advance.'

Chunk 5
Length: 17 characters
Content: 'Sick Leave Policy'

Chunk 6
Length: 54 characters
Content: 'Employees can request sick leave when they are unwell.'

Chunk 7
Length: 59 characters
Content: 'A medical certificate may be required for extended absence.'

Chunk 8
Length: 60 characters
Content: 'Employees must inform their managers when taking sick leave.'

Chunk 9
Length: 21 characters
Content:

### Executing Recursive Splitting

This cell executes the recursive chunking function on the sample document.

- **`document`** – Supplies the input text.
- **`chunk_size=100`** – Sets the maximum chunk length.
- **`recursive_chunking(...)`** – Runs the recursive splitting algorithm.
- **`recursive_chunks`** – Stores the generated chunks.

The output displays each chunk's content and length, followed by validation results.

**What to observe:**
- Every chunk should be non-empty.
- Every chunk should contain no more than 100 characters.
- The chunks should respect meaningful text boundaries whenever possible.
- The number of chunks depends on the input, separator hierarchy, and chunk-size setting.

**Important:** The current validation checks chunk lengths and non-empty chunks, but it does not prove that the original text has been preserved exactly. A separate reconstruction test is needed to verify text preservation.

In [4]:
recursive_chunks = recursive_chunking(document, chunk_size=100)

EXPERIMENT 6: RECURSIVE SPLITTING

Document length: 619 characters
Maximum chunk size: 100 characters
Separator hierarchy: ['\n\n', '\n', '. ', ' ', '']

FINAL CHUNKS
----------------------------------------------------------------------

Chunk 1
Length: 19 characters
Content: 'Annual Leave Policy'

Chunk 2
Length: 52 characters
Content: 'Employees receive 20 days of annual leave each year.'

Chunk 3
Length: 50 characters
Content: 'Leave requests must be approved by their managers.'

Chunk 4
Length: 68 characters
Content: 'Employees should submit leave requests five working days in advance.'

Chunk 5
Length: 17 characters
Content: 'Sick Leave Policy'

Chunk 6
Length: 54 characters
Content: 'Employees can request sick leave when they are unwell.'

Chunk 7
Length: 59 characters
Content: 'A medical certificate may be required for extended absence.'

Chunk 8
Length: 60 characters
Content: 'Employees must inform their managers when taking sick leave.'

Chunk 9
Length: 21 characters
Content: